# Experiment 1 — Authority direction (§4)

Parametrized notebook: set MODEL_KEY in the config cell, then Run All on an A100.

For Exp 2-4, make sure Experiment 1 has already been run for the chosen model (the result bundle is read from Drive).


In [ ]:
# Install
!pip install -q git+https://github.com/YOURUSER/mech_spoof.git

In [ ]:
# Auth + Drive
import os
from google.colab import drive, userdata
try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    pass
try:
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
except Exception:
    pass
drive.mount('/content/drive')

In [ ]:
# Config (EDIT ME)
from pathlib import Path
from mech_spoof.configs import MODEL_CONFIGS

MODEL_KEY = 'qwen'            # qwen | llama3 | mistral | gemma | phi3
EXPERIMENT = 1
DRIVE_ROOT = Path('/content/drive/MyDrive/mech_spoof_results')

slug = MODEL_CONFIGS[MODEL_KEY].slug
exp_dirname = {1: 'exp1_authority', 2: 'exp2_conflict', 3: 'exp3_refusal', 4: 'exp4_attacks'}[EXPERIMENT]
OUT_DIR = DRIVE_ROOT / slug / exp_dirname
OUT_DIR.mkdir(parents=True, exist_ok=True)
print('OUT_DIR =', OUT_DIR)

In [ ]:
# Run the experiment
if EXPERIMENT == 1:
    from mech_spoof.experiments.exp1_authority import run_experiment_1
    result = run_experiment_1(MODEL_KEY, OUT_DIR)
elif EXPERIMENT == 2:
    from mech_spoof.experiments.exp2_conflict import run_experiment_2
    result = run_experiment_2(MODEL_KEY, OUT_DIR, exp1_dir=DRIVE_ROOT / slug / 'exp1_authority')
elif EXPERIMENT == 3:
    from mech_spoof.experiments.exp3_refusal import run_experiment_3
    result = run_experiment_3(MODEL_KEY, OUT_DIR, exp1_dir=DRIVE_ROOT / slug / 'exp1_authority')
elif EXPERIMENT == 4:
    from mech_spoof.experiments.exp4_attacks import run_experiment_4
    result = run_experiment_4(
        MODEL_KEY, OUT_DIR,
        exp1_dir=DRIVE_ROOT / slug / 'exp1_authority',
        exp3_dir=DRIVE_ROOT / slug / 'exp3_refusal',
    )
print('done:', OUT_DIR)

In [ ]:
# Inspect + plot (Exp 1 only — per-model plot)
if EXPERIMENT == 1:
    from mech_spoof.viz import plot_layer_accuracy
    plot_layer_accuracy(result.accuracies, MODEL_KEY, OUT_DIR / 'layer_accuracy.png')
    print({l: round(a, 3) for l, a in result.accuracies.items()})